# Medical Inventory - K-Means Inventory Segmentation

This notebook performs unsupervised product segmentation using `src.clustering`.


## 1. Import Modules and Load Data


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

REPO_ROOT = Path("..").resolve() if Path("..").joinpath("src").exists() else Path(".").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data_processing import clean_product_data
from src.feature_engineering import prepare_ml_dataset
from src.clustering import (
    prepare_clustering_data,
    scale_clustering_data,
    compute_elbow_inertia,
    plot_elbow_method,
    fit_kmeans_clusters,
    plot_cluster_scatter
)

sns.set_theme(style="whitegrid")

product_data_path = REPO_ROOT / "data" / "Product_Level_Data_Final.csv"
stock = clean_product_data(pd.read_csv(product_data_path))
ml_model = prepare_ml_dataset(stock)


## 2. Clustering Feature Preparation & Scaling


In [ ]:
cluster_df = prepare_clustering_data(ml_model)
cluster_scaled, scaler = scale_clustering_data(cluster_df)
print("Clustering dataset preview:")
display(cluster_df.head())


## 3. Elbow Method for Selecting k


In [ ]:
k_list, inertia = compute_elbow_inertia(cluster_scaled, k_range=range(1, 10))
fig_elbow = plot_elbow_method(k_list, inertia)
plt.show()


## 4. K-Means Fitting & Cluster Profiling


In [ ]:
clustered_df, kmeans_model, sil_score = fit_kmeans_clusters(
    cluster_df, cluster_scaled, n_clusters=4, random_state=5
)

print(f"Silhouette Score (k=4): {sil_score:.4f}")
print("\nCluster Count Breakdown:")
print(clustered_df["Cluster_Label"].value_counts())

display(clustered_df.groupby("Cluster_Label").mean())


## 5. Cluster Visualizations


In [ ]:
fig_scatter = plot_cluster_scatter(clustered_df)
plt.show()
